# Enhanced severity models

This optional notebook compares validation-selected models without changing the original notebooks. Use Python 3.12 and install `experiments/temporal_enhancements/requirements.txt` plus JupyterLab.

Selection uses 2021–2023 for fitting and 2024 for model and decision settings. The selected model is refitted on 2021–2024. The 2025 cohort was previously inspected and is not a fresh external test.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from experiments.temporal_enhancements.run import LABELS, split_data, make_model, fit, metrics
root = Path('.')
evidence = root / 'results/temporal_enhancements'
selection = json.loads((evidence / 'selection.json').read_text())
recorded = json.loads((evidence / 'inspected_test.json').read_text())
comparison = pd.DataFrame([{'Operating point': name, 'Model': r['config']['name'], 'Accuracy (%)': 100*r['accuracy'], 'Macro F1': r['macro_f1'], 'Serious recall (%)': 100*r['recall']['Serious'], 'Fatal recall (%)': 100*r['recall']['Fatal']} for name, r in recorded.items()])
display(comparison.round(4))


,Operating point,Model,Accuracy (%),Macro F1,Serious recall (%),Fatal recall (%)
0,accuracy,baseline_rf,76.9017,0.3202,5.0723,0.0000
1,balanced,cat_5_False,69.5205,0.4247,37.3548,12.2549
2,constrained,cat_5_False,69.0589,0.4204,36.2882,16.6667
3,baseline_lr,baseline_lr,70.6012,0.4172,30.7893,12.2549
4,baseline_rf,baseline_rf,70.8373,0.4269,31.2633,19.6078


## Refit the selected operating point

Choose `balanced`, `constrained` or `accuracy` before running. The highest-accuracy option can sacrifice rare-class detection. The constrained recall requirements apply to validation data only and are not guarantees for 2025.


In [2]:
point = 'balanced'
chosen = selection['selected'][point]
display(chosen)
data = pd.read_csv(root / 'data/final_matched_2021_2025.csv.gz', dtype={'collision_index': str}, float_precision='round_trip')
train, validation, test = split_data(data)
features = json.loads((evidence / 'features.json').read_text())
development = pd.concat([train, validation], ignore_index=True)
model = fit(make_model(chosen['config'], features), chosen['config'], development, features)
weights = np.array(chosen['validation']['weights'])
raw = model.predict_proba(test[features])
predicted = (raw * weights).argmax(axis=1)
display(metrics(test.severity, predicted))


{'config': {'name': 'cat_5_False',
  'family': 'cat',
  'params': {'depth': 5},
  'weighted': False},
 'validation': {'accuracy': 0.7082290362953693,
  'macro_f1': 0.4467491997702506,
  'recall': {'Fatal': 0.15808823529411764,
   'Serious': 0.36649471874232376,
   'Slight': 0.8121081372615115},
  'precision': {'Fatal': 0.17408906882591094,
   'Serious': 0.34884264671498716,
   'Slight': 0.8221403221403222},
  'confusion_matrix': [[43, 126, 103], [76, 1492, 2503], [128, 2659, 12046]],
  'label_order': ['Fatal', 'Serious', 'Slight'],
  'weights': [12.0, 3.0, 1.0]}}

{'accuracy': 0.6952051201342986,
 'macro_f1': 0.4246597720118639,
 'recall': {np.str_('Fatal'): 0.12254901960784313,
  np.str_('Serious'): 0.37354823417871535,
  np.str_('Slight'): 0.7958876972470798},
 'precision': {np.str_('Fatal'): 0.09615384615384616,
  np.str_('Serious'): 0.3476726229869843,
  np.str_('Slight'): 0.8165253346415305},
 'confusion_matrix': [[25, 113, 66], [91, 1576, 2552], [144, 2844, 11651]],
 'label_order': ['Fatal', 'Serious', 'Slight']}

In [3]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
print(classification_report(test.severity, LABELS[predicted], labels=LABELS, zero_division=0))
ConfusionMatrixDisplay.from_predictions(test.severity, LABELS[predicted], labels=LABELS, cmap='Blues', values_format='d')
plt.title('Previously inspected 2025 evaluation')
plt.tight_layout()
plt.show()


              precision    recall  f1-score   support

       Fatal       0.10      0.12      0.11       204
     Serious       0.35      0.37      0.36      4219
      Slight       0.82      0.80      0.81     14639

    accuracy                           0.70     19062
   macro avg       0.42      0.43      0.42     19062
weighted avg       0.71      0.70      0.70     19062



## Interpretation

Use the reproduced LR/RF baselines for comparison. Higher accuracy can coexist with lower Fatal recall. Raw scores are not calibrated fatality probabilities. AADF is typical daily flow, not instantaneous density. These are selected major-road crashes, not every road or a validated pre-crash risk population. See `results/temporal_enhancements/README.md` for the executed results and limitations.
